# Injected Decoder Determinism\n
\n
This notebook checks the deterministic single-qubit injection-decoder kernels used for the corrected `injected_baseline(...)` path.\n
\n
Those kernels now use `U3`-based initialization surrogates in Gemini logical:\n
- `X`: a `U3` parameterization whose initialized state is equivalent to `H|0> = |+>`\n
- `Y`: a `U3` parameterization whose initialized state is equivalent to `S H|0>` under the notebook's tomography convention\n
- `Z`: prepare `|0>` and measure in `Z`\n
\n
In noiseless detector-sampler mode, all detector bits and observable bits should be zero for every shot.

In [1]:
import sys
from pathlib import Path

import numpy as np

PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
for candidate in PROJECT_ROOT_CANDIDATES:
    candidate = candidate.resolve()
    if (candidate / 'demo' / 'msd_utils').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('Could not locate repo root containing demo/msd_utils.')

from bloqade.lanes import GeminiLogicalSimulator
from bloqade.lanes.steane_defaults import steane7_m2dets, steane7_m2obs

from demo.msd_utils.circuits import (
    build_injected_decoder_kernel_map,
    build_task_map,
    make_noisy_steane7_initializer,
)
from demo.msd_utils.core import run_task

In [2]:
SHOTS = 4096

sim = GeminiLogicalSimulator()
noisy_steane7_initialize = make_noisy_steane7_initializer(sim)

injected_decoder_tasks = build_task_map(
    sim,
    build_injected_decoder_kernel_map(),
    m2dets=steane7_m2dets(1),
    m2obs=steane7_m2obs(1),
    noisy_initializer=noisy_steane7_initialize,
    append_measurements=False,
)

list(injected_decoder_tasks)

['X', 'Y', 'Z']

In [3]:
injected_decoder_tasks["Z"].tsim_circuit.diagram(height=500)

In [4]:
results = {}
for basis, task in injected_decoder_tasks.items():
    data = run_task(task, SHOTS, with_noise=False)
    results[basis] = {
        'unique_detectors': np.unique(data.detectors, axis=0).tolist(),
        'unique_observables': np.unique(data.observables, axis=0).tolist(),
        'all_zero_detectors': bool(np.all(data.detectors == 0)),
        'all_zero_observables': bool(np.all(data.observables == 0)),
    }

results

{'X': {'unique_detectors': [[0, 0, 0]],
  'unique_observables': [[0]],
  'all_zero_detectors': True,
  'all_zero_observables': True},
 'Y': {'unique_detectors': [[0, 0, 0]],
  'unique_observables': [[0]],
  'all_zero_detectors': True,
  'all_zero_observables': True},
 'Z': {'unique_detectors': [[0, 0, 0]],
  'unique_observables': [[0]],
  'all_zero_detectors': True,
  'all_zero_observables': True}}

In [5]:
for basis in 'XYZ':
    summary = results[basis]
    print(f'===== {basis} =====')
    print('unique detectors:', summary['unique_detectors'])
    print('unique observables:', summary['unique_observables'])
    print('all-zero detectors:', summary['all_zero_detectors'])
    print('all-zero observables:', summary['all_zero_observables'])
    print()

assert all(results[basis]['all_zero_detectors'] for basis in 'XYZ')
assert all(results[basis]['all_zero_observables'] for basis in 'XYZ')
print('Determinism check passed.')

===== X =====
unique detectors: [[0, 0, 0]]
unique observables: [[0]]
all-zero detectors: True
all-zero observables: True

===== Y =====
unique detectors: [[0, 0, 0]]
unique observables: [[0]]
all-zero detectors: True
all-zero observables: True

===== Z =====
unique detectors: [[0, 0, 0]]
unique observables: [[0]]
all-zero detectors: True
all-zero observables: True

Determinism check passed.
